### Build Constructors Dimension

1. Read silver constructors table
2. Read gold ref_nationality_region table
3. Join the data from constructors with ref_nationality_region using nationality
4. Select the required columns:
    * constructors.constructor_id
    * constructors.constructor_name
    * constructors.nationality
    * ref_nationality_region.region
5. Write the transformed data to gold dim_constructors table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table =F"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
constructors_df = spark.read.table(F"{catalog_name}.{silver_schema}.constructors")
ref_nationality_region_df = spark.read.table(F"{catalog_name}.{gold_schema}.ref_nationality_region")



In [0]:
display(constructors_df)

constructor_id,constructor_name,nationality,timestamp,source_file
cooper-alfa_romeo,Cooper-Alfa Romeo,British,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
de_tomaso-ferrari,De Tomaso-Ferrari,Italian,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
apollon,Apollon,Swiss,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
bellasi,Bellasi,Swiss,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
hesketh,Hesketh,British,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
hrt,HRT,Spanish,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
moda,Andrea Moda,Italian,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
cooper-osca,Cooper-OSCA,British,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
epperly,Epperly,American,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json
sauber,Sauber,Swiss,2026-09-13T19:20:29.461Z,dbfs:/Volumes/formula1/landing/files/constructors.json


In [0]:
dim_constructors_df =(
    constructors_df
        .join(ref_nationality_region_df, 
                constructors_df.nationality == ref_nationality_region_df.nationality, 
                "left")
        .select(constructors_df.constructor_id, 
                constructors_df.constructor_name, 
                constructors_df.nationality, 
                ref_nationality_region_df.region.alias("nationality_region") 
        )
)         
    

In [0]:
(
    dim_constructors_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))

constructor_id,constructor_name,nationality,nationality_region
cooper-alfa_romeo,Cooper-Alfa Romeo,British,Europe
de_tomaso-ferrari,De Tomaso-Ferrari,Italian,Europe
apollon,Apollon,Swiss,Europe
bellasi,Bellasi,Swiss,Europe
hesketh,Hesketh,British,Europe
hrt,HRT,Spanish,Europe
moda,Andrea Moda,Italian,Europe
cooper-osca,Cooper-OSCA,British,Europe
epperly,Epperly,American,North America
sauber,Sauber,Swiss,Europe
